In [3]:
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import os, sys
import pandas as pd
from huggingface_hub import snapshot_download

DATA_PATH = os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon/data")
STACK_PATH = os.path.join(os.getenv("MLAB"), "projects/brcameta/projects/sig_recon/scripts/stack")

In [2]:
# 1. Writing Prompt/Query Data
k562_full_data = sc.read(os.path.join(DATA_PATH, "scgpt/k562/perturb_processed.h5ad"))
rpe1_full_data = sc.read(os.path.join(DATA_PATH, "scgpt/rpe1/perturb_processed.h5ad"))

In [18]:
unique_genes = np.array(sorted(list(rpe1_full_data.obs.gene.unique())))

In [28]:
# Change adata.var
k562_full_data.var['ensembl_id'] = k562_full_data.var.index.copy()
# Then set gene names as index, if there are duplicates we append a numeric
k562_full_data.var.index = k562_full_data.var['gene_name'].astype(str)
k562_full_data.var_names_make_unique()
k562_full_data.var.drop('gene_name', axis=1, inplace=True)

rpe1_full_data.var['ensembl_id'] = rpe1_full_data.var.index.copy()
# Then set gene names as index, if there are duplicates we append a numeric
rpe1_full_data.var.index = rpe1_full_data.var['gene_name'].astype(str)
rpe1_full_data.var_names_make_unique()
rpe1_full_data.var.drop('gene_name', axis=1, inplace=True)

In [65]:
# Changing for raw too
if k562_full_data.raw is not None:
    # Get the raw data
    raw_adata = k562_full_data.raw.to_adata()
    raw_adata.var['ensembl_id'] = raw_adata.var.index.copy()
    raw_adata.var_names = raw_adata.var['gene_name'].astype(str)
    raw_adata.var_names_make_unique()
    raw_adata.var.drop('gene_name', axis=1, inplace=True)
    
    k562_full_data.raw = raw_adata

# For rpe1
if rpe1_full_data.raw is not None:
    raw_adata = rpe1_full_data.raw.to_adata()
    raw_adata.var['ensembl_id'] = raw_adata.var.index.copy()
    raw_adata.var_names = raw_adata.var['gene_name'].astype(str)
    raw_adata.var_names_make_unique()
    raw_adata.var.drop('gene_name', axis=1, inplace=True)
    
    rpe1_full_data.raw = raw_adata

In [67]:
k562_full_data.write(os.path.join(DATA_PATH, "stack/k562_all.h5ad"))
rpe1_full_data.write(os.path.join(DATA_PATH, "stack/rpe1_all.h5ad"))

In [68]:
k562_ctrl_data = k562_full_data[k562_full_data.obs.gene == "non-targeting"]
rpe1_ctrl_data = rpe1_full_data[rpe1_full_data.obs.gene == "non-targeting"]

k562_ctrl_data.write(os.path.join(DATA_PATH, "stack/k562_dmso.h5ad"))
rpe1_ctrl_data.write(os.path.join(DATA_PATH, "stack/rpe1_dmso.h5ad"))

In [2]:
k562_ctrl_data = sc.read(os.path.join(DATA_PATH, "stack/k562_dmso.h5ad"))

In [2]:
k562_nontargeting_data = sc.read("/restricted/projectnb/agedisease/CBMrepositoryData/perturbational_data/replogle_2022/k562_stack_pred/non-targeting.h5ad")

In [10]:
k562_nontargeting_data.X[1:10,1:10].toarray()

array([[0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 3., 0.],
       [0., 0., 0., 0., 0., 0., 0., 2., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [13]:
overlap_gene_names = list(set(k562_ctrl_data.var_names) & set(k562_nontargeting_data.var_names))

In [31]:
k562_nontargeting_data[k562_nontargeting_data.obs_names[1:10], overlap_gene_names[1:10]].X.toarray()

array([[ 0.,  1.,  0.,  2., 11.,  3.,  7.,  1.,  2.],
       [ 0.,  0.,  0.,  2.,  4.,  0.,  1.,  0.,  0.],
       [ 0.,  0.,  0.,  1.,  8.,  0.,  3.,  0.,  1.],
       [ 0.,  0.,  0.,  1.,  9.,  1.,  5.,  0.,  0.],
       [ 0.,  0.,  0.,  1.,  4.,  0.,  5.,  0.,  0.],
       [ 0.,  0.,  0.,  0.,  5.,  1.,  3.,  1.,  2.],
       [ 0.,  0.,  0.,  0.,  4.,  0.,  3.,  1.,  2.],
       [ 0.,  0.,  0.,  1.,  4.,  1.,  1.,  1.,  0.],
       [ 0.,  0.,  0.,  1.,  2.,  0.,  1.,  0.,  0.]], dtype=float32)

In [30]:
k562_ctrl_data[k562_nontargeting_data.obs_names[1:10], overlap_gene_names[1:10]].raw.X

array([[0., 0., 2., ..., 1., 0., 1.],
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 3., ..., 0., 0., 0.],
       [0., 0., 3., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(9, 8563), dtype=float32)

In [25]:
k562_ctrl_data

AnnData object with n_obs × n_vars = 10691 × 8563
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'condition', 'cell_type'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id'
    uns: 'log1p'

In [29]:
k562_nontargeting_data.obs_names[1:10]

Index(['AAACCCAAGCGTCTGC-27', 'AAACCCAAGGAGGGTG-47', 'AAACCCAAGTACCCTA-20',
       'AAACCCAAGTGTTCAC-3', 'AAACCCACAACTAGAA-17', 'AAACCCACACATACGT-9',
       'AAACCCACACGGTGTC-19', 'AAACCCACAGTATACC-20', 'AAACCCACATGCGTGC-26'],
      dtype='object', name='cell_barcode')

# Writing Split Data

In [4]:
splits_df = pd.read_csv(os.path.join(DATA_PATH, "sigs/perturb-seq/pb_splits.csv"))

In [8]:
splits_df

,pb,split_1,split_2,split_3,split_4,split_5,split_6,split_7,split_8,split_9,split_10
0,AAAS,False,False,False,False,False,True,False,False,False,False
1,AAMP,False,False,False,False,True,False,False,False,False,False
2,AARS,False,False,False,False,False,False,False,False,False,True
3,AARS2,False,False,False,False,False,True,False,False,False,False
4,AASDHPPT,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
2048,ZNRD1,False,False,False,True,False,False,False,False,False,False
2049,ZRANB2,False,False,False,False,True,False,False,False,False,False
2050,ZRSR2,True,False,False,False,False,False,False,False,False,False
2051,ZW10,True,False,False,False,False,False,False,False,False,False


In [6]:
k562_full_data= sc.read(os.path.join(DATA_PATH, "stack/perturb_seq/k562_all.h5ad"))
rpe1_full_data= sc.read(os.path.join(DATA_PATH, "stack/perturb_seq/rpe1_all.h5ad"))

In [10]:
k562_full_data.obs.gene

cell_barcode
AAACCCAAGAAATCCA-27       NAF1
AAACCCAAGAACTTCC-31       BUB1
AAACCCAAGAAGCCAC-34       UBL5
AAACCCAAGAATAGTC-43    C9orf16
AAACCCAAGACAGCGT-28      TIMM9
                        ...   
TTTGTTGTCTGTCGTC-45    ATP6V1D
TTTGTTGTCTGTCTCG-27      CNOT3
TTTGTTGTCTGTGCGG-44     METTL3
TTTGTTGTCTTGCAGA-14       RPL5
TTTGTTGTCTTTACAC-25     SEC61B
Name: gene, Length: 310385, dtype: category
Categories (2058, object): ['AAAS', 'AAMP', 'AARS', 'AARS2', ..., 'ZRSR2', 'ZW10', 'ZWINT', 'non-targeting']

In [12]:
output_dir = os.path.join(DATA_PATH, "stack/perturb_seq/splits")
os.makedirs(output_dir, exist_ok=True)

pert_col = 'gene'

# Loop through each split (1-10)
for split_num in range(2, 11):
    split_col = f"split_{split_num}"
    
    # Get genes for this split
    genes_in_split = set(splits_df[splits_df[split_col] == True]['pb'].values)
    genes_not_in_split = set(splits_df[splits_df[split_col] == False]['pb'].values)
    
    # --- VERSION 1: K562_all + RPE1 perturbed profiles where pb = TRUE (in split) ---
    
    # Filter rpe1/k562 to only perturbed profiles in the split
    k562_in_split = k562_full_data[k562_full_data.obs[pert_col].isin(genes_in_split)].copy()
    rpe1_in_split = rpe1_full_data[rpe1_full_data.obs[pert_col].isin(genes_in_split)].copy()
    
    # Concatenate
    k562_rpe1_in_split = sc.concat([k562_full_data, rpe1_in_split], 
                                    axis=0, 
                                    label='dataset', 
                                    keys=['k562', 'rpe1'],
                                    index_unique='-')
    rpe1_k562_in_split = sc.concat([rpe1_full_data, k562_in_split], 
                                    axis=0, 
                                    label='dataset', 
                                    keys=['rpe1', 'k562'],
                                    index_unique='-')
    
    del k562_rpe1_in_split, rpe1_k562_in_split  # Free memory
    gc.collect()
    
    # Save
    k562_rpe1_in_split.write(os.path.join(output_dir, f"k562_all_split_10th_{split_num}.h5ad"))
    rpe1_k562_in_split.write(os.path.join(output_dir, f"rpe1_all_split_10th_{split_num}.h5ad"))
    
    # --- VERSION 2: K562_all + RPE1 perturbed profiles where pb = FALSE (not in split) ---
    
    # Filter rpe1/k562 to only perturbed profiles in the split
    k562_in_split = k562_full_data[k562_full_data.obs[pert_col].isin(genes_not_in_split)].copy()
    rpe1_in_split = rpe1_full_data[rpe1_full_data.obs[pert_col].isin(genes_not_in_split)].copy()
    
    # Concatenate
    k562_rpe1_in_split = sc.concat([k562_full_data, rpe1_in_split], 
                                    axis=0, 
                                    label='dataset', 
                                    keys=['k562', 'rpe1'],
                                    index_unique='-')
    rpe1_k562_in_split = sc.concat([rpe1_full_data, k562_in_split], 
                                    axis=0, 
                                    label='dataset', 
                                    keys=['rpe1', 'k562'],
                                    index_unique='-')
    
    del k562_rpe1_in_split, rpe1_k562_in_split  # Free memory
    gc.collect()
    
    # Save
    k562_rpe1_in_split.write(os.path.join(output_dir, f"k562_all_split_90th_{split_num}.h5ad"))
    rpe1_k562_in_split.write(os.path.join(output_dir, f"rpe1_all_split_90th_{split_num}.h5ad"))
    
    # Print summary
    print(f"  Split {split_num}: {len(genes_in_split)} genes in split, {len(genes_not_in_split)} genes out")

print("\nAll splits generated successfully!")

MemoryError: Unable to allocate 4.04 GiB for an array with shape (1085486360,) and data type float32